# 04 — Two-Tower Retrieval + FAISS ANN

This notebook implements neural retrieval.

Architecture:

```text
session history -> Session Tower -> session embedding
item ID         -> Item Tower    -> item embedding
                                  ↓
                             similarity
                                  ↓
                              Top-K items
```

The item side can be precomputed and indexed, which is why Two-Tower models are widely used for large-scale candidate generation.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import faiss

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load processed data and recreate temporal training split

In [ ]:
event_files = sorted(PROCESSED_DIR.glob("otto_train_*_sessions.parquet"))
valid_files = sorted(PROCESSED_DIR.glob("otto_validation_*_sessions.parquet"))

if not event_files or not valid_files:
    raise FileNotFoundError("Run Notebook 02 first.")

events = pd.read_parquet(event_files[-1])
valid_examples = pd.read_parquet(valid_files[-1])

session_meta = (
    events.groupby("session")
    .agg(end_ts=("ts", "max"))
    .reset_index()
    .sort_values("end_ts")
    .reset_index(drop=True)
)

split_idx = int(len(session_meta) * 0.90)
train_session_ids = set(session_meta.iloc[:split_idx]["session"])
train_events = events[events["session"].isin(train_session_ids)].copy()

print(train_events.shape, valid_examples.shape)


## 3. Build the item vocabulary

Index 0 is reserved for padding and unknown items.


In [ ]:
unique_items = train_events["aid"].unique()

item_to_idx = {
    int(aid): idx + 1
    for idx, aid in enumerate(unique_items)
}

idx_to_item = np.zeros(
    len(item_to_idx) + 1,
    dtype=np.int64,
)

for aid, idx in item_to_idx.items():
    idx_to_item[idx] = aid

N_ITEMS = len(item_to_idx) + 1

print(f"Items including padding: {N_ITEMS:,}")


## 4. Build next-item training examples

From a session `[A, B, C, D]`, examples can be:

- `[A] -> B`
- `[A, B] -> C`
- `[A, B, C] -> D`

To control training size locally, only the latest few prediction points are used from each session.


In [ ]:
MAX_HISTORY = 30
MAX_EXAMPLES_PER_SESSION = 5

train_sorted = train_events.sort_values(
    ["session", "ts", "position"]
)

examples = []

for _, group in train_sorted.groupby("session", sort=False):
    sequence = [
        item_to_idx[int(aid)]
        for aid in group["aid"].tolist()
        if int(aid) in item_to_idx
    ]

    if len(sequence) < 2:
        continue

    start = max(
        1,
        len(sequence) - MAX_EXAMPLES_PER_SESSION,
    )

    for target_pos in range(start, len(sequence)):
        history = sequence[
            max(0, target_pos - MAX_HISTORY):target_pos
        ]
        target = sequence[target_pos]

        if history:
            examples.append((history, target))

print(f"Training examples: {len(examples):,}")


## 5. PyTorch Dataset and DataLoader

In [ ]:
class TwoTowerDataset(Dataset):
    def __init__(self, examples, max_history):
        self.examples = examples
        self.max_history = max_history

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        history, target = self.examples[idx]

        padded = (
            [0] * (self.max_history - len(history))
            + history[-self.max_history:]
        )

        return (
            torch.tensor(padded, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
        )


dataset = TwoTowerDataset(
    examples,
    MAX_HISTORY,
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,
    num_workers=0,
)

print(len(dataset))


## 6. Define the Two-Tower model

The session tower:
1. embeds all observed items
2. masks padding
3. mean-pools the history
4. passes it through a small projection network
5. L2-normalizes the result

The item tower embeds and projects each candidate item independently.

Normalized dot product then behaves like cosine similarity.


In [ ]:
class TwoTowerModel(nn.Module):
    def __init__(self, n_items, embedding_dim=64):
        super().__init__()

        self.item_embedding = nn.Embedding(
            n_items,
            embedding_dim,
            padding_idx=0,
        )

        self.session_projection = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim),
        )

        self.item_projection = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

    def encode_session(self, histories):
        mask = (histories != 0).float().unsqueeze(-1)

        embedded = self.item_embedding(histories)

        summed = (embedded * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1.0)

        pooled = summed / counts
        projected = self.session_projection(pooled)

        return nn.functional.normalize(
            projected,
            dim=-1,
        )

    def encode_items(self, item_ids):
        embedded = self.item_embedding(item_ids)
        projected = self.item_projection(embedded)

        return nn.functional.normalize(
            projected,
            dim=-1,
        )

    def forward(self, histories, target_items):
        return (
            self.encode_session(histories),
            self.encode_items(target_items),
        )


model = TwoTowerModel(
    n_items=N_ITEMS,
    embedding_dim=64,
).to(DEVICE)

model


## 7. Train with in-batch negatives

For a batch of B examples, each session's true item is positive and the other B-1 target items become negatives.

This avoids explicitly sampling a large set of negatives for every training example.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

criterion = nn.CrossEntropyLoss()
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for histories, target_items in loader:
        histories = histories.to(DEVICE)
        target_items = target_items.to(DEVICE)

        session_vectors, item_vectors = model(
            histories,
            target_items,
        )

        # Shape: [batch_size, batch_size]
        logits = session_vectors @ item_vectors.T

        # The correct pair for row i is column i.
        labels = torch.arange(
            logits.size(0),
            device=DEVICE,
        )

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"loss={running_loss / len(loader):.4f}"
    )


## 8. Precompute item embeddings

At serving time we do not want to run the item tower for the entire catalog on every request.

We compute item vectors once and put them into a retrieval index.


In [ ]:
model.eval()

item_embedding_batches = []
BATCH_SIZE = 8192

with torch.no_grad():
    for start in range(1, N_ITEMS, BATCH_SIZE):
        end = min(start + BATCH_SIZE, N_ITEMS)

        ids = torch.arange(
            start,
            end,
            dtype=torch.long,
            device=DEVICE,
        )

        vectors = model.encode_items(ids)

        item_embedding_batches.append(
            vectors.cpu().numpy().astype("float32")
        )

item_embeddings = np.vstack(item_embedding_batches)

print(item_embeddings.shape)


## 9. Build a FAISS index

Because embeddings are normalized, inner product is equivalent to cosine similarity.

`IndexFlatIP` is exact search. This is ideal for validating the model locally. Later we can compare approximate indexes such as HNSW/IVF for larger-scale serving.


In [ ]:
embedding_dim = item_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dim
)

faiss_index.add(
    item_embeddings
)

print(f"Indexed items: {faiss_index.ntotal:,}")


## 10. Encode a validation session

In [ ]:
def encode_session(observed_aids):
    indices = [
        item_to_idx.get(int(aid), 0)
        for aid in observed_aids[-MAX_HISTORY:]
    ]

    padded = (
        [0] * (MAX_HISTORY - len(indices))
        + indices
    )

    tensor = torch.tensor(
        [padded],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        vector = model.encode_session(tensor)

    return (
        vector
        .cpu()
        .numpy()
        .astype("float32")
    )


## 11. Two-Tower recommendation

In [ ]:
global_popularity = (
    train_events["aid"]
    .value_counts()
    .index.astype(int)
    .tolist()
)

def recommend_two_tower(row, k=20):
    query = encode_session(
        row.observed_aids
    )

    # Retrieve extra candidates so filtering does not leave too few.
    scores, indices = faiss_index.search(
        query,
        k * 3,
    )

    recommendations = []

    for faiss_idx in indices[0]:
        # FAISS row zero maps to model vocabulary index 1.
        model_idx = int(faiss_idx) + 1
        aid = int(idx_to_item[model_idx])

        if aid != 0 and aid not in recommendations:
            recommendations.append(aid)

        if len(recommendations) >= k:
            break

    if len(recommendations) < k:
        for aid in global_popularity:
            if aid not in recommendations:
                recommendations.append(aid)

            if len(recommendations) >= k:
                break

    return recommendations[:k]


## 12. Evaluate candidate recall

For retrieval, Recall@100 or Recall@200 is usually more important than perfect ordering at 20.

The retriever's job is to make sure the relevant item reaches the downstream ranker.


In [ ]:
def evaluate_retrieval(
    examples,
    recommend_fn,
    k=100,
    max_examples=10_000,
):
    hits = []
    reciprocal_ranks = []

    subset = examples.head(max_examples)

    for row in subset.itertuples(index=False):
        recs = recommend_fn(row, k=k)

        if row.target_aid in recs:
            rank = recs.index(row.target_aid) + 1
            hits.append(1.0)
            reciprocal_ranks.append(1.0 / rank)
        else:
            hits.append(0.0)
            reciprocal_ranks.append(0.0)

    return {
        f"Recall@{k}": float(np.mean(hits)),
        f"MRR@{k}": float(np.mean(reciprocal_ranks)),
    }


results = []

for k in [20, 50, 100, 200]:
    metrics = evaluate_retrieval(
        valid_examples,
        recommend_two_tower,
        k=k,
    )
    results.append({"k": k, **metrics})

pd.DataFrame(results)


## 13. Save model and FAISS index

In [ ]:
torch.save(
    {
        "state_dict": model.state_dict(),
        "item_to_idx": item_to_idx,
        "idx_to_item": idx_to_item,
        "max_history": MAX_HISTORY,
    },
    MODEL_DIR / "two_tower.pt",
)

faiss.write_index(
    faiss_index,
    str(MODEL_DIR / "two_tower_items.faiss"),
)

print("Saved Two-Tower model and FAISS index.")


## Next

Notebook 05 combines multiple candidate generators and introduces Learning-to-Rank.

```text
co-visitation ┐
Item2Vec      ├─> candidate union -> ranking features -> LambdaMART -> Top-20
BPR           │
Two-Tower     ┘
```
